# TREC-SAR — thermal person detector for the aerial spotter (free Colab GPU)

Trains a YOLO11 person detector on **HIT-UAV** (Suo et al., *Scientific Data* 2023): 2,898 drone thermal images,
classes Person / Car / Bicycle / OtherVehicle / DontCare. We keep **Person** only and keep empty images as negatives.

Runtime → Change runtime type → **T4 GPU**. Total time ≈ 30–60 min for 50 epochs.

*Why aerial data?* HIT-UAV and HERIDAL are drone viewpoints, so this model is the **drone's** detector (the aerial cue
in TREC-SAR). The UGV's ground-level camera uses a COCO-pretrained model in the 3D demo — see `perception/README.md`.

In [ ]:
!pip -q install ultralytics requests
import torch, ultralytics; print(torch.cuda.is_available(), ultralytics.__version__)

## 1. Get the code and the dataset
The dataset is archived on Zenodo (DOI 10.5281/zenodo.7633134). If the automatic download fails, download the zip from the Zenodo page in a browser and upload it to `/content/hit-uav.zip`.

In [ ]:
REPO = "https://github.com/mustafarkhan/trec-sar.git"   # change if your repo name differs
!git clone -q $REPO /content/trec-sar || echo "clone failed - upload perception/voc_to_yolo.py manually"

import os, requests, zipfile, glob
ZIP = "/content/hit-uav.zip"
if not os.path.exists(ZIP):
    rec = requests.get("https://zenodo.org/api/records/7633134", timeout=60).json()
    zips = [f for f in rec.get("files", []) if f["key"].endswith(".zip")]
    print("files in record:", [f["key"] for f in rec.get("files", [])])
    url = zips[0]["links"]["self"]
    with requests.get(url, stream=True, timeout=600) as r, open(ZIP, "wb") as f:
        for chunk in r.iter_content(1 << 20):
            f.write(chunk)
zipfile.ZipFile(ZIP).extractall("/content/hit-uav")
ann = [d for d in glob.glob("/content/hit-uav/**/Annotations", recursive=True) if "normal" in d.lower()] \
      or glob.glob("/content/hit-uav/**/Annotations", recursive=True)
print(ann)

## 2. Convert VOC → person-only YOLO

In [ ]:
import pathlib
voc = pathlib.Path(ann[0]).parent
imgs = next(p for p in [voc / "JPEGImages", voc / "images", voc / "Images"] if p.exists())
splits = voc / "ImageSets" / "Main"
!python /content/trec-sar/perception/voc_to_yolo.py --images "$imgs" --annotations "{ann[0]}" \
    --splits "$splits" --classes Person --out /content/datasets/hituav_person

## 3. Train
`yolo11n` is small enough for a Jetson Orin Nano at real-time rates. Thermal images are single-channel, so colour augmentation is switched off.

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")
model.train(data="/content/datasets/hituav_person/data.yaml", epochs=50, imgsz=640, batch=32,
            hsv_h=0.0, hsv_s=0.0, hsv_v=0.3, fliplr=0.5, mosaic=1.0, patience=15,
            project="/content/runs", name="hituav_person", seed=0)

## 4. Evaluate on the held-out test split
For search and rescue, **recall** matters more than precision: a missed person costs a life, a false alarm costs a minute. Report recall at a low confidence threshold as well as mAP.

In [ ]:
import json
best = YOLO("/content/runs/hituav_person/weights/best.pt")
out = {}
for conf in (0.25, 0.10):
    m = best.val(data="/content/datasets/hituav_person/data.yaml", split="test", conf=conf, verbose=False)
    out[f"conf_{conf}"] = {"precision": float(m.box.mp), "recall": float(m.box.mr),
                           "mAP50": float(m.box.map50), "mAP50_95": float(m.box.map)}
print(json.dumps(out, indent=2))
json.dump(out, open("/content/runs/hituav_person/test_metrics.json", "w"), indent=2)

## 5. Export for edge deployment
ONNX runs anywhere; on a Jetson, convert the ONNX to TensorRT on the device itself (`yolo export format=engine` there).

In [ ]:
best.export(format="onnx", imgsz=640, simplify=True)
from google.colab import files
files.download("/content/runs/hituav_person/weights/best.pt")
files.download("/content/runs/hituav_person/test_metrics.json")

## 6. Put the numbers in the README
Copy `test_metrics.json` into `results/perception_metrics.json` and fill the perception table in the README.
**Do not** report validation-split numbers as test numbers. Limitations to state honestly: HIT-UAV was flown at high altitude
over mostly urban and campus scenes, and not winter forest; performance under dense canopy is unknown.